# 05 — Formulating the Burgers' Equation as a PINN Problem

> *"Burgers' equation is the simplest nontrivial PDE in nonlinear fluid mechanics — and the richest pedagogical playground in computational physics."* — folklore

The 1-D viscous Burgers' equation has been the canonical first benchmark for PINNs since Raissi et al. (2019). It is simple enough to write on a napkin, yet contains every essential difficulty: nonlinear advection, dissipative diffusion, formation of near-shocks, sharp transition layers, and a clean exact solution against which a PINN can be benchmarked to machine precision.

This notebook performs the **complete theoretical setup** for the Burgers PINN: derivation of the equation from physical principles, mathematical analysis of its solution structure, the Cole–Hopf transformation, regularity in the vanishing-viscosity limit, choice of initial and boundary conditions for the standard benchmark, and finally the explicit construction of the PINN composite loss. After this notebook the next step is implementation — code.

Throughout we maintain rigour: the Burgers' equation is treated as an object of mathematical physics first, and a target for a neural network second.

## Table of Contents

1. Physical origin: from Navier–Stokes to Burgers
2. The equation, its dimensionless form, and the Reynolds number
3. The inviscid limit: characteristics and shock formation
4. The viscous equation: smoothing and the Cole–Hopf transformation
5. Energy estimates and well-posedness
6. The benchmark problem (Raissi 2019): domain, IC, BC
7. The exact solution as reference
8. Constructing the PINN ansatz
9. Constructing the composite loss
10. Sampling collocation points
11. Anticipated pathologies and mitigations
12. Expected results and success criteria
13. From theory to code — what comes next

## 1. Physical Origin — From Navier–Stokes to Burgers

Begin with the incompressible Navier–Stokes equations for the velocity field $\mathbf{u}(\mathbf{x},t):\mathbb{R}^3\times[0,T]\to\mathbb{R}^3$:

$$
\partial_t \mathbf{u} + (\mathbf{u}\cdot\nabla)\mathbf{u} \;=\; -\frac{1}{\rho}\nabla p + \nu\,\Delta \mathbf{u},\qquad \nabla\cdot\mathbf{u} = 0.
$$

Project onto a single spatial direction and discard incompressibility and pressure to obtain a *scalar* model retaining the two essential features — nonlinear self-advection and viscous diffusion:

$$
\boxed{\partial_t u + u\,\partial_x u \;=\; \nu\,\partial_x^2 u,\qquad u : \mathbb{R}\times[0,T]\to\mathbb{R},\;\nu>0.}
$$

This is the **1-D viscous Burgers' equation**, introduced by J. M. Burgers (1939, 1948) as a toy model for turbulence. Despite its simplicity it captures:

* convective steepening of waveforms ($u\,\partial_x u$);
* viscous dissipation regularizing them ($\nu\,\partial_x^2 u$);
* the formation and propagation of *shock-like transition layers* whose width scales as $\nu$.

Burgers' equation is therefore the simplest setting in which the *competition* between nonlinearity and dissipation can be studied, and so the simplest setting in which a PDE solver — including a PINN — can be stressed against this competition.

## 2. Dimensionless Form and the Reynolds Number

Choose a characteristic velocity $U$, length $L$, and time $L/U$. Define $\tilde u = u/U$, $\tilde x = x/L$, $\tilde t = tU/L$. Substituting,

$$
\partial_{\tilde t}\tilde u + \tilde u\,\partial_{\tilde x}\tilde u \;=\; \frac{\nu}{UL}\,\partial_{\tilde x}^2 \tilde u \;=\; \frac{1}{\mathrm{Re}}\,\partial_{\tilde x}^2\tilde u,
$$

where $\mathrm{Re} = UL/\nu$ is the **Reynolds number**. In dimensionless form,

$$
\partial_t u + u\,\partial_x u \;=\; \frac{1}{\mathrm{Re}}\,\partial_x^2 u.
$$

The benchmark setting in Raissi et al. (2019) uses $\nu = 0.01/\pi$ in physical units on $x\in[-1,1]$, which corresponds to $\mathrm{Re} = \pi/0.01 \approx 314$. This is a *moderately stiff* regime — high enough that an internal transition layer forms, low enough that a single $\tanh$-MLP can resolve it with care.

We use the small parameter $\nu \ll 1$ interchangeably with $\mathrm{Re}^{-1}$ throughout.

## 3. The Inviscid Limit — Characteristics and Shock Formation

Set $\nu = 0$:

$$
\partial_t u + u\,\partial_x u \;=\; 0.
$$

This is a first-order quasilinear PDE solvable by the *method of characteristics*.

### 3.1 Characteristic curves

Along a characteristic curve $X(t)$ defined by $\dot X(t) = u(X(t),t)$, the solution is constant:

$$
\frac{d}{dt}u(X(t),t) = \partial_t u + \dot X(t)\,\partial_x u = \partial_t u + u\,\partial_x u = 0.
$$

Hence if $u(X(0),0) = u_0(\xi)$ then $u$ along the characteristic equals $u_0(\xi)$, and the characteristic is the *straight line*

$$
X(t) = \xi + u_0(\xi)\,t.
$$

### 3.2 Wave breaking

Two characteristics from $\xi_1, \xi_2$ with $u_0(\xi_1) > u_0(\xi_2)$ and $\xi_1 < \xi_2$ collide at time

$$
t^\ast = \frac{\xi_2 - \xi_1}{u_0(\xi_1) - u_0(\xi_2)}.
$$

The first such collision occurs at

$$
t_b \;=\; -\frac{1}{\min_{\xi}\,u_0'(\xi)},
$$

*the breaking time*, beyond which the classical solution becomes multi-valued — physically impossible. For $u_0(x) = -\sin(\pi x)$ (the Raissi benchmark), $u_0'(0) = -\pi$, giving $t_b = 1/\pi \approx 0.318$.

### 3.3 Weak solutions and the Rankine–Hugoniot condition

Beyond $t_b$, the solution is defined only in the weak (distributional) sense, and a *shock* — a discontinuity — propagates with speed

$$
s = \frac{F(u^+)-F(u^-)}{u^+ - u^-} = \frac{u^+ + u^-}{2},\qquad F(u) = \tfrac{1}{2}u^2,
$$

the *Rankine–Hugoniot* shock speed for Burgers' flux. Among the (infinite) weak solutions, the *entropy* (Lax, Kruzhkov) condition selects the unique physically admissible one.

### 3.4 What viscosity does

Reintroduce $\nu > 0$. The viscous term *smooths* the would-be shock into a *transition layer* of width $\mathcal{O}(\nu / |\Delta u|)$ where $\Delta u$ is the jump magnitude. For Raissi's $\nu = 0.01/\pi$ and $\Delta u \approx 2$, the layer width is roughly $\nu/|\Delta u| \approx 1.6\cdot 10^{-3}$ — three orders of magnitude smaller than the domain length. **This is the central challenge for any PDE solver, including PINNs: resolving the layer.**

## 4. The Cole–Hopf Transformation

Cole (1951) and Hopf (1950) independently discovered a remarkable *linearization*: the substitution

$$
u(x,t) \;=\; -2\nu\,\frac{\partial_x \varphi(x,t)}{\varphi(x,t)},
$$

transforms the nonlinear Burgers equation into the **linear heat equation**

$$
\partial_t \varphi \;=\; \nu\,\partial_x^2 \varphi.
$$

### 4.1 Verification

Compute $\partial_t u$, $\partial_x u$, and $\partial_x^2 u$ in terms of $\varphi$ using the quotient rule and substitute. After algebraic simplification using $\partial_t \varphi = \nu\partial_x^2\varphi$, one obtains the Burgers equation for $u$ identically. (We omit the four lines of calculus.)

### 4.2 Solving via the heat kernel

The Cauchy problem for the heat equation has the explicit solution

$$
\varphi(x,t) \;=\; \frac{1}{\sqrt{4\pi\nu t}}\int_{\mathbb{R}} \exp\!\left(-\frac{(x-\xi)^2}{4\nu t}\right)\varphi_0(\xi)\,d\xi,
$$

with initial datum determined from $u_0$ via

$$
\varphi_0(\xi) \;=\; \exp\!\left(-\frac{1}{2\nu}\int_0^\xi u_0(\eta)\,d\eta\right).
$$

Hence the exact viscous Burgers solution is

$$
u(x,t) \;=\; -2\nu\,\frac{\partial_x \varphi(x,t)}{\varphi(x,t)} \;=\; \frac{\displaystyle\int_\mathbb{R}\frac{x-\xi}{t}\,e^{-G(x,\xi,t)/(2\nu)}\,d\xi}{\displaystyle\int_\mathbb{R} e^{-G(x,\xi,t)/(2\nu)}\,d\xi},
$$

$$
G(x,\xi,t) := \frac{(x-\xi)^2}{2t} + \int_0^\xi u_0(\eta)\,d\eta.
$$

This integral representation is the reference solution we will compare a PINN against. For the benchmark $u_0(x) = -\sin(\pi x)$ on $[-1,1]$, it is computed by quadrature with extreme precision.

### 4.3 The vanishing-viscosity limit

As $\nu \to 0^+$ the integral representation localizes (Laplace's method) around the minimum of $G$ in $\xi$, recovering the inviscid characteristic solution together with the entropy condition. The Cole–Hopf solution is *the* canonical example of how the vanishing-viscosity limit selects entropy solutions — a fact central to the modern theory of hyperbolic conservation laws.

## 5. Energy Estimates and Well-Posedness

### 5.1 $L^2$ energy decay

Multiply the viscous Burgers equation by $u$ and integrate over $\mathbb{R}$ (or a periodic interval):

$$
\frac{1}{2}\frac{d}{dt}\int u^2\,dx + \int u^2\,\partial_x u\,dx = \nu\int u\,\partial_x^2 u\,dx.
$$

The convective term is $\int \partial_x(\tfrac{1}{3}u^3)\,dx = 0$ by integration by parts; the diffusion term gives $-\nu\int (\partial_x u)^2\,dx$. Hence

$$
\frac{d}{dt}\|u(\cdot,t)\|_{L^2}^2 = -2\nu\|\partial_x u(\cdot,t)\|_{L^2}^2 \le 0.
$$

Energy is monotone decreasing — a strong stability property. The PINN solution should respect this; *energy increase in time is a clear diagnostic of training failure*.

### 5.2 Maximum principle

For $\nu > 0$, the viscous Burgers equation satisfies a maximum principle:

$$
\min_x u_0(x) \;\le\; u(x,t) \;\le\; \max_x u_0(x) \qquad \forall (x,t).
$$

For $u_0(x) = -\sin(\pi x)$, $|u(x,t)|\le 1$. PINN solutions that violate this bound by more than a small percentage are physically nonsensical.

### 5.3 Existence and uniqueness

For $u_0 \in L^2(\mathbb{R})\cap L^\infty(\mathbb{R})$ (or smooth periodic), the viscous Burgers equation has a unique classical solution $u \in C^\infty(\mathbb{R}\times(0,\infty))$ for all $t > 0$. Well-posedness is *not* in doubt — what is hard is *numerical* resolution of the layer.

## 6. The Benchmark Problem (Raissi et al. 2019)

We adopt the canonical Burgers benchmark from Raissi, Perdikaris, Karniadakis (2019), §4.1.1.

### 6.1 PDE

$$
\boxed{\;\partial_t u + u\,\partial_x u \;-\; \frac{0.01}{\pi}\,\partial_x^2 u \;=\; 0,\qquad (x,t)\in (-1,1)\times(0,1].\;}
$$

### 6.2 Initial condition

$$
u(x,0) \;=\; -\sin(\pi x),\qquad x\in[-1,1].
$$

Note that $u_0'(0) = -\pi$, so the inviscid breaking time is $t_b = 1/\pi \approx 0.318$.

### 6.3 Boundary conditions

Homogeneous Dirichlet:

$$
u(-1,t) \;=\; u(1,t) \;=\; 0,\qquad t\in[0,1].
$$

### 6.4 Why this problem?

It exhibits, in a 1+1-dimensional domain accessible to a single MLP:

* a smooth solution at $t=0$;
* monotone steepening of the negative-velocity region into the positive-velocity region;
* formation of an extremely narrow transition layer near $x=0$ for $t \gtrsim 0.3$, with width $\mathcal{O}(\nu)$;
* an exact analytical (integral-form) solution against which to benchmark to $10^{-6}$ precision.

It is small enough to train in minutes on a CPU and large enough to expose every PINN pathology of Notebook 3.

## 7. The Exact Solution as Reference

The Cole–Hopf representation with $u_0(x) = -\sin(\pi x)$ on $[-1,1]$ and homogeneous Dirichlet BC admits a Fourier-series representation (Basdevant et al. 1986):

$$
u(x,t) \;=\; -2\nu\,\frac{\partial_x \varphi(x,t)}{\varphi(x,t)},\qquad \varphi(x,t) = a_0 + 2\sum_{n=1}^\infty a_n\,e^{-n^2\pi^2\nu t}\cos(n\pi x),
$$

with

$$
a_0 = \int_0^1 e^{-(1-\cos\pi\xi)/(2\pi\nu)}\,d\xi,\qquad a_n = \int_0^1 e^{-(1-\cos\pi\xi)/(2\pi\nu)}\cos(n\pi\xi)\,d\xi.
$$

These coefficients are computed once by adaptive quadrature; the series converges exponentially due to the $e^{-n^2\pi^2\nu t}$ factor. Truncation at $n=200$ gives single-precision (and at $n=400$ double-precision) accuracy.

We will use this exact solution to compute relative $L^2$ and $L^\infty$ errors of the trained PINN.

## 8. Constructing the PINN Ansatz

### 8.1 The network

Let $u_\theta : \mathbb{R}^2 \to \mathbb{R}$ be a fully-connected $\tanh$-MLP of depth $L = 8$ and width $W = 20$:

$$
u_\theta(x,t) \;=\; \big(W^{(L+1)} \,\sigma\,W^{(L)}\,\sigma\,W^{(L-1)}\cdots\sigma\,W^{(1)}\big)\binom{x}{t} + b^{(L+1)},\qquad \sigma = \tanh.
$$

(Bias vectors $b^{(\ell)}$ are added at each layer; suppressed for clarity.) Total parameter count $\sim 3000$ — comfortably in the over-parameterized regime where UAT and NTK theory apply.

### 8.2 Input normalization

Map inputs to $[-1,1]^2$:

$$
\hat x = x,\qquad \hat t = 2t - 1.
$$

Helps optimizer conditioning. (Already $x\in[-1,1]$; only $t$ needs rescaling.)

### 8.3 Output post-processing (optional hard constraint)

A high-performance variant enforces ICs and BCs by construction:

$$
u_\theta(x,t) \;=\; (1-t)\,u_0(x) + t(1-x^2)\,\tilde u_\theta(x,t),
$$

with $\tilde u_\theta$ the raw MLP. Then $u_\theta(x,0) = u_0(x)$ exactly and $u_\theta(\pm 1,t) = 0$ exactly. The boundary and initial loss terms vanish identically. We will explore both the soft-constraint and hard-constraint variants in code (Notebook 02_burgers).

### 8.4 Smoothness check

$\tanh \in C^\infty(\mathbb{R})$. Hence $u_\theta \in C^\infty(\mathbb{R}^2)$ for any $\theta$, satisfying the regularity demanded by the second-order operator $\partial_x^2$.

## 9. Constructing the Composite Loss

Define the PDE *residual* at any point $(x,t)$:

$$
r_\theta(x,t) \;:=\; \partial_t u_\theta(x,t) + u_\theta(x,t)\,\partial_x u_\theta(x,t) - \nu\,\partial_x^2 u_\theta(x,t),\qquad \nu = 0.01/\pi.
$$

All three derivatives are computed by automatic differentiation (Notebook 2). $r_\theta(x,t)\equiv 0$ would mean $u_\theta$ solves the PDE exactly at $(x,t)$.

### 9.1 Residual loss

$$
\mathcal{L}_r(\theta) \;=\; \frac{1}{N_r}\sum_{i=1}^{N_r}\big|r_\theta(x^r_i,t^r_i)\big|^2,\qquad (x^r_i,t^r_i)\in (-1,1)\times(0,1].
$$

We use $N_r = 10{,}000$ Latin Hypercube samples (Raissi's choice).

### 9.2 Initial loss

$$
\mathcal{L}_0(\theta) \;=\; \frac{1}{N_0}\sum_{i=1}^{N_0}\big|u_\theta(x^0_i,0) - (-\sin(\pi x^0_i))\big|^2,\qquad x^0_i \in [-1,1].
$$

$N_0 = 100$ uniform samples on $x\in[-1,1]$.

### 9.3 Boundary loss

$$
\mathcal{L}_b(\theta) \;=\; \frac{1}{N_b}\sum_{i=1}^{N_b}\big(|u_\theta(-1,t^b_i) - 0|^2 + |u_\theta(1,t^b_i) - 0|^2\big),\qquad t^b_i\in[0,1].
$$

$N_b = 100$ uniform samples on $t\in[0,1]$.

### 9.4 Composite

$$
\boxed{\;\mathcal{L}_{\text{PINN}}(\theta) \;=\; \mathcal{L}_r(\theta) + \mathcal{L}_0(\theta) + \mathcal{L}_b(\theta).\;}
$$

Equal weighting is the published baseline; we will explore NTK-based weighting and LR-annealing variants in code.

### 9.5 Variations to be explored in code

* **Hard-constraint variant.** $\mathcal{L}_0 = \mathcal{L}_b = 0$ by construction; only $\mathcal{L}_r$ remains.
* **Adaptive weighting.** Update $\lambda_r, \lambda_0, \lambda_b$ via the LR-annealing rule of Wang et al. (2021).
* **RAR sampling.** Periodically resample $\{(x^r_i,t^r_i)\}$ from a density $\propto |r_\theta|^2$ to focus on the transition layer.
* **Causal weighting.** Up-weight small-$t$ residuals relative to large-$t$ to enforce temporal causality.

## 10. Sampling Collocation Points

### 10.1 Latin Hypercube

For $N_r = 10000$ interior points, sample from LHS$([−1,1]\times[0,1])$. Latin Hypercube guarantees marginal uniformity in each axis and significantly lower discrepancy than i.i.d. Uniform — see McKay et al. (1979).

### 10.2 Sobol' alternative

Sobol' quasi-random sequences yield even lower discrepancy in low dimensions. Either works for Burgers; LHS is the historical baseline.

### 10.3 Boundary and initial

Sample $N_0 = N_b = 100$ uniformly. Because these are 1-D arrays, the count can be modest.

### 10.4 Adaptive refinement

After an initial Adam phase, recompute the residual field on a dense $1000\times 200$ grid, sample $K$ new points from the residual-weighted density, and union with the existing set. Repeat every $N_{\text{refresh}}$ epochs. This dramatically improves layer resolution for Burgers.

## 11. Anticipated Pathologies and Mitigations

From the theory in Notebook 3 we predict the following difficulties when training the Burgers PINN.

### 11.1 Underresolved transition layer

**Symptom.** $u_\theta$ is smooth and monotone but does not reach the steep gradient $\partial_x u \sim -1/\nu$ at $x=0, t\sim 0.7$.

**Cause.** Spectral bias — high-frequency content of the layer is learned exponentially slowly.

**Mitigation.** Fourier feature embedding of inputs with $\sigma_B \sim 10$, or SIREN activations; RAR sampling to concentrate points in the layer.

### 11.2 Boundary violation

**Symptom.** $u_\theta(\pm 1,t) \ne 0$ during early Adam phases; PDE residual loss decreases at the expense of BC compliance.

**Cause.** Gradient pathology — $|g_r| \gg |g_b|$.

**Mitigation.** Increase $\lambda_b$ adaptively, or use the hard-constraint output transform.

### 11.3 Energy creation

**Symptom.** $\|u_\theta(\cdot,t)\|_{L^2}^2$ increases somewhere in $[0,1]$, violating the energy estimate.

**Cause.** PDE residual not driven low enough, or boundary leak.

**Mitigation.** Train longer; switch to L-BFGS; check for boundary violation.

### 11.4 Maximum-principle violation

**Symptom.** $\max_{x,t}|u_\theta(x,t)| > 1$ (the maximum of $|u_0|$).

**Cause.** Overshooting near the layer.

**Mitigation.** RAR resampling around the violation region; activation function with bounded image (tanh helps).

### 11.5 L-BFGS divergence

**Symptom.** L-BFGS stage shows oscillating loss.

**Cause.** Adam stage stopped too early — initial point not yet in a quadratic basin.

**Mitigation.** Extend Adam to at least $5\cdot 10^4$ iterations; check that $\mathcal{L}_{\text{PINN}} < 10^{-3}$ before switching.

## 12. Expected Results and Success Criteria

On the Raissi benchmark a properly trained PINN should achieve:

| Quantity | Target |
|---|---|
| Final $\mathcal{L}_{\text{PINN}}$ | $< 10^{-5}$ |
| Relative $L^2$ error vs. exact | $< 10^{-3}$ |
| Relative $L^\infty$ error vs. exact | $< 10^{-2}$ |
| Energy monotonicity | $\|u_\theta(\cdot,t)\|_{L^2}$ non-increasing within $10^{-3}$ tolerance |
| Maximum-principle | $\|u_\theta\|_\infty \le 1 + 10^{-3}$ |
| Visual: shock-layer position | Transition near $x=0$ for $t\gtrsim 0.5$ |
| Train time (CPU) | $\sim 5$–$10$ min Adam + $\sim 1$–$2$ min L-BFGS |

These figures are achievable with the architecture and hyperparameters described in §8–§9. Beating them requires the advanced variants of Notebook 4.

## 13. From Theory to Code — What Comes Next

We have now assembled the complete theoretical scaffolding:

1. **Notebook 1** justified why a continuous neural ansatz can in principle solve PDEs (UAT in Sobolev norms; the calculus of variations).
2. **Notebook 2** explained how its derivatives are computed exactly and cheaply (automatic differentiation on smooth composites).
3. **Notebook 3** characterized the optimization landscape, its pathologies, and the canonical Adam→L-BFGS pipeline.
4. **Notebook 4** surveyed structural extensions (cPINN, VPINN, XPINN, B-PINN) for problems where the vanilla PINN cannot succeed.
5. **This notebook** specialized the entire framework to the 1-D viscous Burgers equation: derived the equation, analyzed its solution structure, gave the Cole–Hopf exact reference, and wrote down the PINN composite loss explicitly.

The next notebook (`02_burgers_equation_pinn.ipynb`) implements this in PyTorch with the following structure, all already justified by the theory above:

1. Define the $\tanh$-MLP $u_\theta$;
2. Sample LHS collocation points;
3. Compute the residual $r_\theta$ by two nested AD calls;
4. Form and weight the composite loss;
5. Train with Adam (5e4 iterations) → L-BFGS (5e3 iterations) at double precision;
6. Evaluate against the Cole–Hopf series solution;
7. Plot $u_\theta(x,t)$ surfaces, slices at $t \in \{0.25, 0.5, 0.75\}$, and the residual field.

The theory is complete. The code follows.

## References

1. **Burgers, J. M.** *Mathematical examples illustrating relations occurring in the theory of turbulent fluid motion.* Trans. Roy. Neth. Acad. Sci. 17 (1939).
2. **Burgers, J. M.** *A mathematical model illustrating the theory of turbulence.* Adv. Appl. Mech. 1 (1948), 171–199.
3. **Cole, J. D.** *On a quasi-linear parabolic equation occurring in aerodynamics.* Quart. Appl. Math. 9 (1951), 225–236.
4. **Hopf, E.** *The partial differential equation $u_t + u u_x = \mu u_{xx}$.* Comm. Pure Appl. Math. 3 (1950), 201–230.
5. **Basdevant, C., Deville, M., Haldenwang, P., Lacroix, J. M., Ouazzani, J., Peyret, R., Orlandi, P., Patera, A. T.** *Spectral and finite difference solutions of the Burgers equation.* Computers and Fluids 14 (1986), 23–41.
6. **Raissi, M., Perdikaris, P., Karniadakis, G. E.** *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.* J. Comp. Phys. 378 (2019), 686–707.
7. **Whitham, G. B.** *Linear and Nonlinear Waves.* Wiley, 1974. (Chapters 2–4 on conservation laws, characteristics, and shock formation.)
8. **Evans, L. C.** *Partial Differential Equations.* AMS GSM, 2010. (Chapter 3 for entropy / vanishing viscosity.)
9. **LeVeque, R. J.** *Finite Volume Methods for Hyperbolic Problems.* Cambridge UP, 2002.
10. **McKay, M. D., Beckman, R. J., Conover, W. J.** *A comparison of three methods for selecting values of input variables in the analysis of output from a computer code.* Technometrics 21 (1979), 239–245.